# Association Rules — Hands-On Exercises

## Scenario: What should a warung put next to the cashier?

A small Indonesian warung wants to increase basket size. The owner has a pile of
receipts and wants to know which products are bought together: **Indomie with
telur**, **kopi sachet with gula**, **rokok with korek api**, and **roti with
susu**.

In this notebook you will:

1. generate a reproducible synthetic warung dataset,
2. encode baskets and compute support / confidence / lift / leverage / conviction,
3. mine frequent itemsets with Apriori and FP-Growth,
4. generate and interpret association rules,
5. verify the anti-monotone property and see why pruning matters.

Every cell runs top-to-bottom. Cells marked **TODO** contain a small challenge;
run them first, then change the values and observe what happens.

## Exercise 1 — Generate and inspect baskets

**Goal:** Understand what one transaction looks like.

**TODO:** After running the cell, add one extra item to a basket and check that
duplicates are removed by `encode_transactions` later.

**Hint:** A basket is simply a list of product names. The generator plants at
most one of the four target associations per basket.

In [ ]:
from pathlib import Path
import math
import sys

# Make the notebook work from the repository root or from the module folder.
cwd = Path.cwd()
for candidate in (cwd, cwd / "02_unsupervised_learning" / "06_association_rules"):
    if (candidate / "implementation.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd
from implementation import (
    Apriori,
    apriori,
    FPGrowth,
    PLANTED_ASSOCIATIONS,
    WARUNG_PRODUCTS,
    confidence,
    conviction,
    encode_transactions,
    fpgrowth,
    generate_rules,
    generate_warung_transactions,
    leverage,
    lift,
    one_hot_encode,
    rule_metrics,
    support,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

transactions = generate_warung_transactions(n_transactions=800, random_state=42)

print(f"Baskets       : {len(transactions)}")
print(f"Catalog size  : {len(WARUNG_PRODUCTS)}")
print(f"Average basket: {sum(len(b) for b in transactions) / len(transactions):.2f} items")
print("\nFirst 8 baskets:")
for basket in transactions[:8]:
    print("  -", ", ".join(basket))

## Exercise 2 — Encode transactions and look at item frequencies

One-hot encoding turns every basket into a boolean row. This is the format used
by many libraries (including mlxtend).

**TODO:** Which three products are bought most often? Do they match your
intuition about a warung?

**Hint:** `one_hot_encode(transactions)` returns a DataFrame; `.sum()` counts
how many baskets contain each item.

In [ ]:
encoded = encode_transactions(transactions)
matrix = one_hot_encode(transactions)

print(f"Encoded baskets : {len(encoded)}")
print(f"One-hot shape   : {matrix.shape}")
print(f"Duplicates removed in first basket? {len(transactions[0]) == len(encoded[0])}")
print("\nTop 10 items by number of baskets:")
print(matrix.sum().sort_values(ascending=False).head(10).to_string())

## Exercise 3 — Compute the five metrics by hand

For a rule **X → Y**:

```
support(X → Y) = P(X ∪ Y)
confidence      = P(X ∪ Y) / P(X)
lift            = confidence / P(Y)
leverage        = P(X ∪ Y) - P(X) * P(Y)
conviction      = (1 - P(Y)) / (1 - confidence)
```

**TODO:** Compute `support(Indomie Goreng → Telur Ayam)` manually from the
one-hot matrix and compare it with the function output.

**Hint:** The pair support is the number of rows where both columns are `True`,
divided by the number of baskets.

In [ ]:
left, right = "Indomie Goreng", "Telur Ayam"

manual_support = (
    (matrix[left] & matrix[right]).sum() / len(matrix)
)

print(f"Manual pair support       : {manual_support:.6f}")
print(f"support({{left, right}})     : {support(transactions, [left, right]):.6f}")
print(f"confidence({left} -> {right}): {confidence(transactions, left, right):.3f}")
print(f"lift                      : {lift(transactions, left, right):.3f}")
print(f"leverage                  : {leverage(transactions, left, right):+.3f}")
print(f"conviction                : {conviction(transactions, left, right):.3f}")

print("\nAll metrics at once:")
print(rule_metrics(transactions, left, right))

### How to read lift

| Lift | Meaning |
|------|---------|
| `> 1` | The products appear together **more** often than chance (positive association). |
| `= 1` | Independent. |
| `< 1` | They appear together **less** often than chance (negative association). |

Conviction is `inf` for a perfect rule (confidence 1) and is close to 1 when the
consequent is very common.

**Question:** Is Indomie + telur a positive association? What about a random
pair such as sabun + es krim?

## Exercise 4 — Mine frequent itemsets with Apriori

Apriori builds candidates level by level:

1. count single items,
2. **join** frequent `(k-1)`-itemsets that share a prefix,
3. **prune** any candidate that contains an infrequent `(k-1)`-subset,
4. count the survivors and keep those above `min_support`.

**TODO:** Try `min_support=0.10`. How many itemsets disappear? Why is that
expected?

**Hint:** A higher `min_support` keeps fewer items; by anti-monotonicity, every
superset of an infrequent itemset is also infrequent.

In [ ]:
min_support = 0.03
apriori_model = Apriori(min_support=min_support).fit(transactions)

print(f"min_support       : {min_support}")
print(f"Frequent itemsets : {len(apriori_model.frequent_itemsets_)}")
print("\nLevel-by-level statistics:")
print(pd.DataFrame(apriori_model.level_stats_).to_string(index=False))

print("\nTop 15 itemsets by support:")
apriori_df = apriori_model.to_dataframe().copy()
apriori_df["itemset_str"] = apriori_df["itemsets"].map(
    lambda s: "{" + ", ".join(sorted(s)) + "}"
)
print(apriori_df[["support", "itemset_str"]].head(15).to_string(index=False))

## Exercise 5 — FP-Growth should find exactly the same itemsets

FP-Growth avoids candidate generation. It builds an **FP-tree** (a compressed
prefix tree) and mines **conditional pattern bases** recursively.

**TODO:** Run both algorithms with `max_len=2` and confirm they still agree.
Then try `max_len=1`: only single items should remain.

**Hint:** `fpgrowth(transactions, min_support=..., max_len=...)` returns a
dictionary, just like `apriori(...)`.

In [ ]:
fp_model = FPGrowth(min_support=min_support).fit(transactions)

print(f"Apriori itemsets  : {len(apriori_model.frequent_itemsets_)}")
print(f"FP-Growth itemsets: {len(fp_model.frequent_itemsets_)}")
print(f"Identical          : {apriori_model.frequent_itemsets_ == fp_model.frequent_itemsets_}")

# The dictionary returned by the convenience function must match too.
our_apriori = apriori_model.frequent_itemsets_
our_fpgrowth = fpgrowth(transactions, min_support=min_support)
assert our_apriori == our_fpgrowth

max_len_result = apriori(transactions, min_support=min_support, max_len=2)
print(f"With max_len=2    : {len(max_len_result)} itemsets, "
      f"largest size = {max(len(s) for s in max_len_result)}")

## Exercise 6 — Generate and filter association rules

Rules are generated from every frequent itemset with at least two items. Each
itemset is split into a non-empty antecedent and consequent; only rules with
`confidence >= min_confidence` are kept.

**TODO:** Raise `min_confidence` to `0.80`. Which planted rules survive? What
happens at `1.0`?

**Hint:** Perfect rules have confidence 1 and infinite conviction (unless the
consequent appears in every basket).

In [ ]:
rules = generate_rules(apriori_model.frequent_itemsets_, min_confidence=0.50)

print(f"Rules with confidence >= 0.50: {len(rules)}")
print("\nColumns:", list(rules.columns))

display_columns = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "leverage",
    "conviction",
]
top_rules = rules.sort_values("lift", ascending=False).head(12).copy()
top_rules["antecedents"] = top_rules["antecedents"].map(
    lambda s: ", ".join(sorted(s))
)
top_rules["consequents"] = top_rules["consequents"].map(
    lambda s: ", ".join(sorted(s))
)
print("\nTop 12 rules by lift:")
print(top_rules[display_columns].to_string(index=False))

strong_rules = rules[(rules["lift"] > 1.5) & (rules["confidence"] > 0.60)]
print(f"\nRules with lift > 1.5 and confidence > 0.60: {len(strong_rules)}")

## Exercise 7 — Did we recover the planted associations?

The generator deliberately over-represents four pairs. A useful sanity check is
to confirm that each pair has positive lift.

**TODO:** Add a fifth pair of your own to a copy of the transactions (for
example `"Teh Celup"` and `"Gula Pasir"`) and compare its lift with the planted
pairs.

**Hint:** `rule_metrics(transactions, left, right)` returns every metric as a
dictionary.

In [ ]:
print("Planted associations:")
for left, right in PLANTED_ASSOCIATIONS:
    metrics = rule_metrics(transactions, left, right)
    print(
        f"  {left:16s} -> {right:12s}  "
        f"support={metrics['support']:.3f}  "
        f"confidence={metrics['confidence']:.3f}  "
        f"lift={metrics['lift']:.2f}  "
        f"leverage={metrics['leverage']:+.3f}  "
        f"conviction={metrics['conviction']:.2f}"
    )

print("\nA random pair for comparison:")
random_metrics = rule_metrics(transactions, "Sabun Mandi", "Es Krim")
print(f"  lift={random_metrics['lift']:.2f}, confidence={random_metrics['confidence']:.3f}")

## Exercise 8 — How does `min_support` change the result?

Lower support finds more itemsets (and more rules), but also more noise and a
longer runtime.

**TODO:** Predict the number of itemsets for `min_support=0.01` before running
the cell. Then run it and check.

**Hint:** At support `s`, an itemset must appear in at least `ceil(s * n)`
baskets. With 800 baskets and `s=0.01`, that is only 8 baskets.

In [ ]:
rows = []
for min_sup in [0.01, 0.02, 0.03, 0.05, 0.10, 0.20]:
    model = Apriori(min_support=min_sup).fit(transactions)
    n_rules = len(generate_rules(model.frequent_itemsets_, min_confidence=0.50))
    rows.append(
        {
            "min_support": min_sup,
            "min_baskets": int(math.ceil(min_sup * len(transactions))),
            "frequent_itemsets": len(model.frequent_itemsets_),
            "rules": n_rules,
        }
    )

print(pd.DataFrame(rows).to_string(index=False))

## Exercise 9 — Anti-monotonicity and pruning in action

If an itemset is frequent, **all of its subsets must be frequent**. Equivalently,
if an itemset is infrequent, **none of its supersets can be frequent**.

The Apriori implementation records `joined_candidates`, `pruned_candidates`,
`candidates` and `frequent` for every level. The gap between joined and
surviving candidates is the effect of the subset check.

**TODO:** Find a `min_support` where `pruned_candidates` is greater than zero and
explain which subset caused the prune.

**Hint:** Start at `0.02`; the level-3 and level-4 statistics are usually the
most interesting.

In [ ]:
pruning_model = Apriori(min_support=0.02).fit(transactions)
stats = pd.DataFrame(pruning_model.level_stats_)
print(stats.to_string(index=False))

total_pruned = int(stats["pruned_candidates"].sum())
print(f"\nTotal candidates removed by subset pruning: {total_pruned}")

# Direct check of the anti-monotone property.
frequent = set(pruning_model.frequent_itemsets_)
violations = 0
for itemset in frequent:
    if len(itemset) == 1:
        continue  # the only proper subset is the empty set
    for item in itemset:
        if frozenset(itemset - {item}) not in frequent:
            violations += 1

print(f"Subsets that are not frequent (should be 0): {violations}")
assert violations == 0

## Summary

You have practiced the full association-rule workflow:

- ✅ encoded baskets into sets and a one-hot matrix,
- ✅ computed support, confidence, lift, leverage and conviction,
- ✅ mined frequent itemsets with **Apriori** (join + anti-monotone prune),
- ✅ mined the same itemsets with **FP-Growth** (FP-tree + conditional bases),
- ✅ generated rules filtered by `min_confidence`,
- ✅ recovered the planted warung associations and inspected lift,
- ✅ measured how `min_support` controls the number of itemsets and rules.

### Next steps

1. Change `generate_warung_transactions` probabilities and see which planted
   pairs disappear.
2. Use `max_len=2` to generate only pairwise rules — useful for shelf placement.
3. Sort rules by **leverage** instead of lift. Which rule has the largest absolute
   effect?
4. Read `README.md` for the theory and `sklearn_example.py` for an mlxtend
   comparison.